In [83]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [84]:
df = pd.read_spss("MBSAQIP_PUF_MAIN.sav")

adiposity_columns = ['BMI_HIGH_BAR', 'BMI', 'WGT_HIGH_BAR', 'WGT_CLOSEST', 'VENOUS_STASIS', 'HGT']
behavioral_columns = ['DIABETES', 'NBHTN_MEDS', 'HYPERLIPIDEMIA', 'SLEEP_APNEA', 'RENAL_INSUFFICIENCY', 'GERD', 'DIALYSIS', 'CREATININE', 'HEMO', 'ALBUMIN']
metabolic_columns = ['FUNSTATPRESURG', 'SMOKER', 'PREVIOUS_SURGERY']
socioenvironmental_columns = ['AGE', 'ASACLASS', 'RACE_PUF', 'SEX', 'HISPANIC', 'OPYEAR', 'SURGSPECIALTY_BAR']
target_column = ['BMI_CLOSEST30D']

feature_columns = adiposity_columns + behavioral_columns + metabolic_columns + socioenvironmental_columns
df = df.replace('', np.nan)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = df[columns].dropna()
print(df.head())

    BMI_HIGH_BAR    BMI  WGT_HIGH_BAR  WGT_CLOSEST VENOUS_STASIS     HGT  \
0          52.91  49.68        262.00       246.00            No   59.00   
7          46.59  42.15        279.98       253.31            No   65.00   
16         57.00  56.73        388.80       387.00            No   69.25   
17         36.07  35.98        210.00        95.00            No  162.50   
18         44.52  42.57        228.00       218.00            No   60.00   

            DIABETES NBHTN_MEDS HYPERLIPIDEMIA SLEEP_APNEA  \
0   Yes, non-insulin          2            Yes          No   
7                 No          2             No         Yes   
16                No  3 or more            Yes         Yes   
17                No          1             No         Yes   
18                No          2            Yes         Yes   

   RENAL_INSUFFICIENCY GERD DIALYSIS  CREATININE  HEMO  ALBUMIN  \
0                   No  Yes       No        0.82   6.2      4.4   
7                   No   No       No

In [85]:
binary_columns = ['VENOUS_STASIS', 'HYPERLIPIDEMIA', 'SLEEP_APNEA', 'RENAL_INSUFFICIENCY', 'GERD', 'DIALYSIS', 'SMOKER', 'PREVIOUS_SURGERY']
for col in binary_columns:
    print(col, df[col].unique())
df[binary_columns] = df[binary_columns].replace({'Yes': 1, 'No': 0})

df['DIABETES'] = df['DIABETES'].replace({'Yes, non-insulin': 1, 'Yes, insulin': 1, 'No': 0})

df['NBHTN_MEDS'] = df['NBHTN_MEDS'].replace({'0': 0, '1': 1, '2': 2, '3 or more': 3})

df = df[df['FUNSTATPRESURG'] != 'Unknown']
df['FUNSTATPRESURG'] = df['FUNSTATPRESURG'].replace({'Independent': 0, 'Partially dependent': 1, 'Totally dependent': 2})

df = df[df['ASACLASS'] != 'None assigned']
df['ASACLASS'] = df['ASACLASS'].replace({'ASA I - Normal/Healthy': 1, 'ASA II - Mild systemic disease': 2, 'ASA III - Severe systemic disease': 3, 'ASA IV - Severe systemic disease threat to life': 4, 'ASA V - Moribund': 5})

df = df[df['RACE_PUF'] != 'Unknown/Not Reported']
race_dummies = pd.get_dummies(df['RACE_PUF'], prefix='RACE').astype(int)
df = pd.concat([df, race_dummies], axis=1)
df = df.drop(columns=['RACE_PUF'])

sex_dummies = pd.get_dummies(df['SEX'], prefix='SEX').astype(int)
df = pd.concat([df, sex_dummies], axis=1)
df = df.drop(columns=['SEX'])

df = df[df['HISPANIC'] != 'Unknown']
df['HISPANIC'] = df['HISPANIC'].replace({'No': 0, 'Yes': 1})

specialty_dummies = pd.get_dummies(df['SURGSPECIALTY_BAR'], prefix='SPECIALTY').astype(int)
df = pd.concat([df, specialty_dummies], axis=1)
df = df.drop(columns=['SURGSPECIALTY_BAR'])

VENOUS_STASIS ['No' 'Yes']
HYPERLIPIDEMIA ['Yes' 'No']
SLEEP_APNEA ['No' 'Yes']
RENAL_INSUFFICIENCY ['No' 'Yes']
GERD ['Yes' 'No']
DIALYSIS ['No' 'Yes']
SMOKER ['No' 'Yes']
PREVIOUS_SURGERY ['No' 'Yes']


/var/folders/70/qt7rxyg13lv618327b11t7jc0000gn/T/ipykernel_96337/746976430.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_columns] = df[binary_columns].replace({'Yes': 1, 'No': 0})
/var/folders/70/qt7rxyg13lv618327b11t7jc0000gn/T/ipykernel_96337/746976430.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DIABETES'] = df['DIABETES'].replace({'Yes, non-insulin': 1, 'Yes, insulin': 1, 'No': 0})
/var/folders/70/qt7rxyg13lv618327b11t7jc0000gn/T/ipykernel_96337/746976430.py:8: FutureWarning: Downcasting behavior in 

In [86]:
race_dummy_cols = [c for c in df.columns if c.startswith('RACE_')]
sex_dummy_cols = [c for c in df.columns if c.startswith('SEX_')]
specialty_dummy_cols = [c for c in df.columns if c.startswith('SPECIALTY_')]

socioenvironmental_columns = ['AGE', 'ASACLASS', 'HISPANIC', 'OPYEAR'] + race_dummy_cols + sex_dummy_cols + specialty_dummy_cols

feature_columns = adiposity_columns + behavioral_columns + metabolic_columns + socioenvironmental_columns

In [87]:
# columns that are continuous (not 0/1)
continuous_columns = ['BMI_HIGH_BAR', 'BMI', 'WGT_HIGH_BAR', 'WGT_CLOSEST', 'HGT', 'CREATININE', 'HEMO', 'ALBUMIN', 'AGE', 'OPYEAR']

scaler = StandardScaler()
df[continuous_columns] = scaler.fit_transform(df[continuous_columns])

target_scaler = StandardScaler()
df[['BMI_CLOSEST30D']] = target_scaler.fit_transform(df[['BMI_CLOSEST30D']])

In [95]:
class Layer:
    def __init__(self, n_inputs, n_neurons, activation):
        self.W = np.random.randn(n_inputs, n_neurons) * np.sqrt(1 / n_inputs)
        self.b = np.zeros((1, n_neurons))
        self.activation = activation

    def forward(self, X):
        self.X = X
        Z = X.dot(self.W) + self.b
        self.Z = Z
        if self.activation == 'relu':
            A = np.maximum(0, Z)
        elif self.activation == 'linear':
            A = Z
        return A

    def backward(self, dA):
        m = self.X.shape[0]
        if self.activation == 'relu':
            dZ = dA * (self.Z > 0).astype(float)
        elif self.activation == 'linear':
            dZ = dA
        dW = self.X.T.dot(dZ) / m
        db = np.sum(dZ, axis=0, keepdims=True) / m
        dX = dZ.dot(self.W.T)
        return dW, db, dX

    def update(self, dW, db, learning_rate):
        self.W = self.W - learning_rate * dW
        self.b = self.b - learning_rate * db

In [96]:
class Network:
    def __init__(self, input_size, hidden_size, output_size):
        self.layer1 = Layer(input_size, hidden_size, 'relu')
        self.layer2 = Layer(hidden_size, output_size, 'linear')

    def forward(self, X):
        A1 = self.layer1.forward(X)
        A2 = self.layer2.forward(A1)
        return A2

    def backward(self, dA):
        dW2, db2, dX2 = self.layer2.backward(dA)
        dW1, db1, dX1 = self.layer1.backward(dX2)
        return dW1, db1, dW2, db2

    def update(self, dW1, db1, dW2, db2, learning_rate):
        self.layer1.update(dW1, db1, learning_rate)
        self.layer2.update(dW2, db2, learning_rate)

In [112]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_loss_derivative(y_true, y_pred):
    return -2 * (y_true - y_pred)

def train(network, X, y, epochs, learning_rate):
    for epoch in range(epochs):
        y_pred = network.forward(X)
        loss = mse_loss(y, y_pred)
        dA = mse_loss_derivative(y, y_pred)
        dW1, db1, dW2, db2 = network.backward(dA)
        network.update(dW1, db1, dW2, db2, learning_rate)
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, loss: {loss:.4f}")

In [113]:
X = df[feature_columns].values
y = df[[target_column[0]]].values  # target_column is a list from your code, so [0] gets the string

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [115]:
n_inputs = X_train.shape[1]
net = Network(input_size=n_inputs, hidden_size=8, output_size=1)

train(net, X_train, y_train, epochs=5000, learning_rate=0.1)

Epoch 0, loss: 1.3527
Epoch 50, loss: 0.1990
Epoch 100, loss: 0.5040
Epoch 150, loss: 0.4962
Epoch 200, loss: 0.4917
Epoch 250, loss: 0.2922
Epoch 300, loss: 0.0899
Epoch 350, loss: 0.0775
Epoch 400, loss: 0.0751
Epoch 450, loss: 0.0736
Epoch 500, loss: 0.0724
Epoch 550, loss: 0.0713
Epoch 600, loss: 0.0705
Epoch 650, loss: 0.0699
Epoch 700, loss: 0.0694
Epoch 750, loss: 0.0691
Epoch 800, loss: 0.0688
Epoch 850, loss: 0.0685
Epoch 900, loss: 0.0683
Epoch 950, loss: 0.0681
Epoch 1000, loss: 0.0679
Epoch 1050, loss: 0.0678
Epoch 1100, loss: 0.0676
Epoch 1150, loss: 0.0675
Epoch 1200, loss: 0.0674
Epoch 1250, loss: 0.0672
Epoch 1300, loss: 0.0671
Epoch 1350, loss: 0.0670
Epoch 1400, loss: 0.0670
Epoch 1450, loss: 0.0669
Epoch 1500, loss: 0.0668
Epoch 1550, loss: 0.0668
Epoch 1600, loss: 0.0667
Epoch 1650, loss: 0.0667
Epoch 1700, loss: 0.0666
Epoch 1750, loss: 0.0666
Epoch 1800, loss: 0.0665
Epoch 1850, loss: 0.0665
Epoch 1900, loss: 0.0665
Epoch 1950, loss: 0.0664
Epoch 2000, loss: 0.066

In [110]:
h1 = net.layer1.forward(X_train)
print((h1 > 0).mean(axis=0))

[0.36567315 0.76941685 0.87406767 0.11974082 0.76694024 0.58329734
 0.35337653 0.26683945]
